In [ ]:
{
 "cells": [
  {
   "cell_type": "markdown",
   "metadata": {},
   "source": [
    "# User Story 1: Nghiên cứu Phân tích Vận hành Lưới điện (Operational Manager)\n",
    "\n",
    "## 1. Bảng ánh xạ Khung thiết kế What - Why - How (Tamara Munzner)\n",
    "\n",
    "| Thành phần | Đặc tả kỹ thuật & Nghiệp vụ |\n",
    "| :--- | :--- |\n",
    "| **WHAT (Dữ liệu)** | - Chuỗi thời gian đa cấp (Giờ, Ngày, Tháng, Mùa).<br>- 6 biến phụ tải điện (HUFL, HULL, MUFL, MULL, LUFL, LULL) & tỷ số tải hữu công/vô công.<br>- Chuỗi nhiệt độ dầu OT và ma trận tương quan trễ. |\n",
    "| **WHY (Tác vụ trừu tượng)** | - **Identify Trend & Seasonality**: Nhận diện quy luật phụ tải cao điểm/thấp điểm qua 24 khung giờ.<br>- **Compare Proportions**: So sánh tỷ trọng tổn thất vô công giữa 3 cấp điện áp.<br>- **Assess Lag Relationship**: Xác định độ trễ quán tính nhiệt của trạm biến áp để điều độ công suất tối ưu. |\n",
    "| **HOW (Mã hóa thị giác & Kênh)** | - **Hourly Profile + Historical Quantile Ribbon (Fan Chart)**: Biểu diễn dải phân vị lịch sử 80% (IQR) và 90% (5%-95%) giảm thiểu bất định.<br>- **100% Stacked Bar/Area Chart**: Đối chiếu tỷ trọng tải hữu ích (Useful) và tải vô ích (Unuseful).<br>- **Lag Cross-Correlation Heatmap**: Phân tích tương quan trễ từ 0h đến 12h. |\n",
    "\n",
    "---\n",
    "\n",
    "## 2. Danh mục Tiêu đề Hành động (Action Titles Catalog)\n",
    "1. **Action Title 1 (Hourly Fan Chart):** *\"Nhiệt độ dầu OT đạt đỉnh vào khung 14h-17h hàng ngày với dải biến thiên lịch sử 90% dao động từ 18°C đến 38°C vào mùa hè.\"*\n",
    "2. **Action Title 2 (Tỷ trọng Tải):** *\"Cấp Cao áp (HUFL) chiếm tỷ trọng công suất hữu ích áp đảo (>55%), trong khi cấp Hạ áp ghi nhận tỷ lệ hao phí vô công cao nhất vào mùa đông.\"*\n",
    "3. **Action Title 3 (Tương quan trễ):** *\"Độ trễ quán tính nhiệt của trạm biến áp đạt cực đại sau 2 - 3 giờ đối với tải Cao áp (HUFL) trước khi nhiệt độ dầu OT tăng vọt.\"*"
   ]
  },
  {
   "cell_type": "code",
   "execution_count": null,
   "metadata": {},
   "outputs": [],
   "source": [
    "import os\n",
    "import json\n",
    "import pandas as pd\n",
    "import numpy as np\n",
    "import plotly.express as px\n",
    "import plotly.graph_objects as go\n",
    "\n",
    "# Thiết lập đường dẫn dữ liệu tương đối từ thư mục notebooks/\n",
    "BASE_DIR = os.path.abspath('..')\n",
    "ARTIFACTS_DIR = os.path.join(BASE_DIR, 'output', 'artifacts', 'ETTh1')\n",
    "\n",
    "# Nạp dữ liệu sạch và các artifacts đã được tiền tính toán\n",
    "clean_df = pd.read_parquet(os.path.join(ARTIFACTS_DIR, 'clean_ett.parquet'))\n",
    "quantiles_df = pd.read_parquet(os.path.join(ARTIFACTS_DIR, 'quantile_bands.parquet'))\n",
    "\n",
    "with open(os.path.join(ARTIFACTS_DIR, 'lag_correlation_matrix.json'), 'r', encoding='utf-8') as f:\n",
    "    lag_df = pd.DataFrame(json.load(f))\n",
    "\n",
    "print(f\"Nạp thành công dataset ETTh1: {clean_df.shape[0]} bản ghi.\")"
   ]
  },
  {
   "cell_type": "code",
   "execution_count": null,
   "metadata": {},
   "outputs": [],
   "source": [
    "# Khám phá 1: Hourly Profile + Historical Quantile Ribbon (Fan Chart)\n",
    "q_summer = quantiles_df[quantiles_df['season'] == 'Summer']\n",
    "\n",
    "fig_fan = go.Figure()\n",
    "\n",
    "# Dải 5% - 95% (Độ chắn sáng alpha = 0.14 theo chuẩn Tufte/DoD)\n",
    "fig_fan.add_trace(go.Scatter(\n",
    "    x=q_summer['hour'], y=q_summer['OT_q95'],\n",
    "    mode='lines', line=dict(width=0), showlegend=False\n",
    "))\n",
    "fig_fan.add_trace(go.Scatter(\n",
    "    x=q_summer['hour'], y=q_summer['OT_q05'],\n",
    "    mode='lines', fill='tonexty',\n",
    "    fillcolor='rgba(30, 58, 138, 0.14)',\n",
    "    name='Dải 90% Lịch sử (5%-95%)'\n",
    "))\n",
    "\n",
    "# Dải IQR 25% - 75% (Độ chắn sáng alpha = 0.28)\n",
    "fig_fan.add_trace(go.Scatter(\n",
    "    x=q_summer['hour'], y=q_summer['OT_q75'],\n",
    "    mode='lines', line=dict(width=0), showlegend=False\n",
    "))\n",
    "fig_fan.add_trace(go.Scatter(\n",
    "    x=q_summer['hour'], y=q_summer['OT_q25'],\n",
    "    mode='lines', fill='tonexty',\n",
    "    fillcolor='rgba(30, 58, 138, 0.28)',\n",
    "    name='Dải 50% IQR (25%-75%)'\n",
    "))\n",
    "\n",
    "# Đường Trung vị (Median)\n",
    "fig_fan.add_trace(go.Scatter(\n",
    "    x=q_summer['hour'], y=q_summer['OT_q50'],\n",
    "    mode='lines+markers',\n",
    "    line=dict(color='#1E3A8A', width=2.5),\n",
    "    name='Nhiệt độ Trung vị (P50)'\n",
    "))\n",
    "\n",
    "fig_fan.update_layout(\n",
    "    title='<b>Action Title:</b> Nhiệt độ dầu OT đạt đỉnh vào khung 14h-17h hàng ngày với dải biến thiên 90% dao động từ 18°C đến 38°C vào mùa hè',\n",
    "    xaxis_title='Khung giờ trong ngày (0 - 23h)',\n",
    "    yaxis_title='Nhiệt độ dầu biến áp OT (°C)',\n",
    "    template='plotly_white',\n",
    "    hovermode='x unified'\n",
    ")\n",
    "fig_fan.show()"
   ]
  },
  {
   "cell_type": "code",
   "execution_count": null,
   "metadata": {},
   "outputs": [],
   "source": [
    "# Khám phá 2: Ma trận Tương quan trễ (Lag Cross-Correlation Heatmap)\n",
    "load_cols = ['HUFL', 'HULL', 'MUFL', 'MULL', 'LUFL', 'LULL']\n",
    "corr_matrix = lag_df[load_cols].values.T\n",
    "\n",
    "fig_lag = px.imshow(\n",
    "    corr_matrix,\n",
    "    x=lag_df['lag_hours'],\n",
    "    y=load_cols,\n",
    "    color_continuous_scale='Viridis',\n",
    "    labels=dict(x='Độ trễ thời gian (Giờ)', y='Nhánh phụ tải', color='Pearson Corr'),\n",
    "    title='<b>Action Title:</b> Độ trễ quán tính nhiệt của trạm biến áp đạt cực đại sau 2 - 3 giờ đối với tải Cao áp (HUFL)'\n",
    ")\n",
    "\n",
    "fig_lag.update_layout(\n",
    "    template='plotly_white',\n",
    "    xaxis=dict(dtick=1)\n",
    ")\n",
    "fig_lag.show()"
   ]
  }
 ],
 "metadata": {
  "language_info": {
   "name": "python"
  }
 },
 "nbformat": 4,
 "nbformat_minor": 2
}